### Importing the file


In [0]:
orders_df = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv("/Volumes/dbacademy/default/ecommerce_data/olist_orders_dataset.csv")
)

In [0]:
display(orders_df)

In [0]:
orders_df.printSchema()

In [0]:
# Explore the dataset
# Total number of records
orders_df.count()

# List of all columns
orders_df.columns

# Data type of each column
orders_df.dtypes

# Schema of the DataFrame
orders_df.printSchema()

# Display the first 5 rows
orders_df.show(5)

# Display the DataFrame in Databricks
display(orders_df)

# View a few important columns
orders_df.select("order_id", "customer_id", "order_status").show(5)

# Basic summary statistics (only for numeric columns)
orders_df.describe().show()

In [0]:
# Total number of rows
orders_df.count()

In [0]:

# Number of unique order IDs
orders_df.select("order_id").distinct().count()

In [0]:
# checking if cust id is all unique or not 
orders_df.select("customer_id").distinct().count()

In [0]:
from pyspark.sql.functions import col, count, when

# Check for missing values in each column
orders_df.select(
    [
        count(when(col(c).isNull(), c)).alias(c)
        for c in orders_df.columns
    ]
).show()

 Insight = there are missing values in the following col :-order_approved_at ,order_delivered_carrier_date, order_delivered_customer_date

In [0]:
#lets investigate the order status col
# Checking the different order statuses
orders_df.select("order_status").distinct().show()

a small insight from the above is that we have an order status as canceled so its very natural that we will have null values in order_delivered_carrier_date, order_delivered_customer_date columns 

In [0]:
# Count orders by status
orders_df.groupBy("order_status").count().show()

In [0]:
#checking for null values in order_delivered_customer_date col where order status is delivered
orders_df.filter(
    (orders_df.order_status == "delivered") &
    (orders_df.order_delivered_customer_date.isNull())
).count()

### Order statuses explain most of the null values

We found these statuses:

delivered,
canceled,
processing,
shipped,
unavailable,
approved,
created,
invoiced,

This explains why many delivery-related columns are null.

For example:

A canceled order will never have a delivery date.
A processing order hasn't been shipped yet.
An unavailable order cannot be delivered.

Insight:

The majority of null values are expected because they correspond to orders that never reached the delivery stage.

I investigated the exception with a simple question

How many delivered orders are missing a delivery date?

The answer is:8

Out of 96,478 delivered orders, only 8 have a missing customer delivery date.

Insight:

Only 8 delivered orders have a missing delivery date, representing a very small data quality issue (less than 0.01% of delivered orders). This suggests the dataset is highly reliable overall, though these records should be reviewed during the Silver layer transformation.

In [0]:

#checking the same for teh other 2 col 
orders_df.filter(
    (orders_df.order_status == "delivered") &
    (orders_df.order_delivered_carrier_date.isNull())
).count()

In [0]:
orders_df.filter(
    (orders_df.order_status == "delivered") &
    (orders_df.order_approved_at.isNull())
).count()

Insight: The validation shows that the Orders dataset has excellent data quality. Out of 96,478 delivered orders, only 14 records are missing the approval timestamp and only 2 are missing the carrier handover timestamp. These are isolated exceptions and indicate a very low level of data quality issues, which can be addressed during the data cleaning phase in the Silver layer.

In [0]:
# Check for duplicate rows

total_rows = orders_df.count()
distinct_rows = orders_df.distinct().count()

print(f"Total Rows: {total_rows}")
print(f"Distinct Rows: {distinct_rows}")
print(f"Duplicate Rows: {total_rows - distinct_rows}")

The Bronze laye ends here lets start the silver layer

In [0]:
# Create the Bronze DataFrame
bronze_orders_df = orders_df

bronze_orders_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bronze_orders") #we have saved this file as a table now

In [0]:
bronze_orders = spark.table("bronze_orders")

display(bronze_orders)

### Silver Layer

The Silver layer contains cleaned and standardized data.

In [0]:
#A delivered order must have an order_delivered_customer_date, We found only 8 records that violate this rule so we can drop those records
from pyspark.sql.functions import col

silver_orders_df = bronze_orders_df.filter(
    ~(
        (col("order_status") == "delivered") &
        (col("order_delivered_customer_date").isNull())
    )
)

In [0]:
#verify
print("Bronze Rows :", bronze_orders.count())
print("Silver Rows :", silver_orders_df.count())

In [0]:
from pyspark.sql.functions import col

silver_orders_df = silver_orders_df.filter(
    ~(
        (col("order_status") == "delivered") &
        (col("order_approved_at").isNull())
    )
)

In [0]:
#verifying 
print("Rows after removing missing approval date:", silver_orders_df.count())

In [0]:
silver_orders_df = silver_orders_df.filter(
    ~(
        (col("order_status") == "delivered") &
        (col("order_delivered_carrier_date").isNull())
    )
)

In [0]:
print("Final Silver Rows:", silver_orders_df.count())

In a real company would we actually delete these 24 records?

The answer is: not always.

Many companies do not delete suspicious records.

### lets save the Silver layer file now


In [0]:
silver_orders_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_orders")

In [0]:
#verifying it 
silver_orders = spark.table("silver_orders")

display(silver_orders)

we have created bronze and silver layer for the order table now lets do the same for the others

In [0]:
customers_df = spark.read.csv(
    "/Volumes/dbacademy/default/ecommerce_data/olist_customers_dataset.csv",
    header=True,
    inferSchema=True
)

In [0]:
customers_df.display()

In [0]:
customers_df.printSchema()

In [0]:
customers_df.count()

In [0]:
#checking if each cust id is unique
customers_df.select("customer_id").distinct().count()

In [0]:
#null values
from pyspark.sql.functions import col, count, when

customers_df.select(
    [
        count(when(col(c).isNull(), c)).alias(c)
        for c in customers_df.columns
    ]
).show()

In [0]:
# checking duplicate rows
total_rows = customers_df.count()
distinct_rows = customers_df.distinct().count()

print(f"Total Rows: {total_rows}")
print(f"Distinct Rows: {distinct_rows}")
print(f"Duplicate Rows: {total_rows - distinct_rows}")

In [0]:
customers_df.select("customer_id", "customer_unique_id").show(10, truncate=False)

### Insight :-
this tables looks very well maintained as ther are no null values along with no duplicated records the schema is on point so we can move forward with creating the bronze and silver layer

In [0]:
customers_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bronze_customers")

In [0]:
bronze_customers = spark.table("bronze_customers")

display(bronze_customers)

In [0]:
silver_customers_df = bronze_customers

In [0]:
silver_customers_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_customers")

In [0]:
silver_customers = spark.table("silver_customers")

display(silver_customers)

In [0]:
order_items_df = spark.read.csv(
    "/Volumes/dbacademy/default/ecommerce_data/olist_order_items_dataset.csv",
    header=True,
    inferSchema=True
)

In [0]:
order_items_df.printSchema()

order_items_df.count()

order_items_df.columns

display(order_items_df)

In [0]:
order_items_df.count()

In [0]:
order_items_df.select("order_id").distinct().count()

In [0]:
order_items_df.select(
    "order_id",
    "order_item_id"
).distinct().count()

In [0]:
order_items_df.count()

### Insight :-
I first checked whether order_id could serve as the primary key by comparing the total number of records with the number of distinct order_id values. Since the total row count (112,650) was higher than the distinct order_id count (98,666), it became clear that order_id was repeated across multiple rows. This indicated that a single order can contain multiple products, meaning order_id alone cannot uniquely identify each record.


### Insight 2:- Identifying the Composite Primary Key
Based on the repeated order_id values, I concluded that the table likely uses a composite primary key. Since order_item_id represents the item number within an order, I combined order_id and order_item_id and checked their uniqueness. The distinct count matched the total number of records, confirming that the combination of these two columns uniquely identifies each row. This validated that (order_id, order_item_id) is the composite primary key for the Order Items table.

In [0]:
# null value 
from pyspark.sql.functions import col, count, when

order_items_df.select(
    [
        count(when(col(c).isNull(), c)).alias(c)
        for c in order_items_df.columns
    ]
).show()

In [0]:
#duplicates
total_rows = order_items_df.count()
distinct_rows = order_items_df.distinct().count()

print(f"Total Rows: {total_rows}")
print(f"Distinct Rows: {distinct_rows}")

In [0]:
#checking for discripency
order_items_df.filter(col("price") < 0).count()  # price should not have a negative value

In [0]:
order_items_df.filter(col("freight_value") < 0).count()

In [0]:
#shipping limit should not be null
order_items_df.filter(col("shipping_limit_date").isNull()).count()

### Bronze table


In [0]:
order_items_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bronze_order_items")

In [0]:
bronze_order_items = spark.table("bronze_order_items")
display(bronze_order_items)

### Since no cleaning was requiredwe will create the Silver DataFrame

In [0]:
silver_order_items_df = bronze_order_items

In [0]:
silver_order_items_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_order_items")

In [0]:
silver_order_items = spark.table("silver_order_items")
display(silver_order_items)

### importing the product tabel

In [0]:
products_df = spark.read.csv(
    "/Volumes/dbacademy/default/ecommerce_data/olist_products_dataset.csv",
    header=True,
    inferSchema=True
)

In [0]:
products_df.printSchema()

products_df.columns

In [0]:
display(products_df)

In [0]:
products_df.count()

In [0]:
products_df.select("product_id").distinct().count()

Since both values match we can confirm product id is the primary key

In [0]:
from pyspark.sql.functions import col, count, when

products_df.select(
    [
        count(when(col(c).isNull(), c)).alias(c)
        for c in products_df.columns
    ]
).show()

### Insights :-
During null value analysis, I observed that the missing values occurred in groups rather than randomly. Specifically, 610 products were missing all descriptive attributes, while only 2 products were missing all physical dimension attributes. This pattern suggests incomplete product information rather than isolated data entry errors, so the business context should be considered before deciding whether to remove or impute these records.

I identified missing values in several product attributes. Since these values represent incomplete information from the source system rather than invalid data, I chose to retain the records without imputation or deletion. This approach preserves data integrity while allowing downstream reporting or business teams to decide how to handle incomplete product information based on their specific use cases.

In [0]:
total_rows = products_df.count()
distinct_rows = products_df.distinct().count()

print(f"Total Rows: {total_rows}")
print(f"Distinct Rows: {distinct_rows}")

In [0]:
#lets check some business rules as well
products_df.filter(col("product_weight_g") < 0).count() #product weight shouldnt be negative

In [0]:
products_df.filter(
    (col("product_length_cm") < 0) |
    (col("product_height_cm") < 0) |
    (col("product_width_cm") < 0)
).count()

In [0]:
products_df.filter(col("product_photos_qty") < 0).count()

In [0]:
#bronze layer
products_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bronze_products")

In [0]:
bronze_products = spark.table("bronze_products")
display(bronze_products)

In [0]:
#silver layer
silver_products_df = bronze_products

In [0]:
silver_products_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_products")

In [0]:
silver_products = spark.table("silver_products")
display(silver_products)

The Products dataset contained 32,951 records, with product_id successfully validated as the primary key. During data quality analysis, I identified grouped null values in descriptive and dimensional attributes, indicating incomplete source information rather than random missing data. Since no business rules were available to justify imputation or deletion, I retained these records to preserve data integrity. Further validation confirmed there were no duplicate records or negative values in product dimensions, weight, or photo count. The dataset therefore met the required quality standards and was promoted from the Bronze layer to the Silver layer without transformation.

### Lets import the payments table

In [0]:
payments_df = spark.read.csv(
    "/Volumes/dbacademy/default/ecommerce_data/olist_order_payments_dataset.csv",
    header=True,
    inferSchema=True
)

In [0]:
payments_df.printSchema()

payments_df.columns
display(payments_df)

In [0]:
payments_df.count()

In [0]:
# checking the primary key
payments_df.select("order_id").distinct().count()

orderid itself is not the primary key lets check if this table has a composit key or not


In [0]:
payments_df.select(
    "order_id",
    "payment_sequential"
).distinct().count()

In [0]:
from pyspark.sql.functions import col, count, when

payments_df.select(
    [
        count(when(col(c).isNull(), c)).alias(c)
        for c in payments_df.columns
    ]
).show()

In [0]:
total_rows = payments_df.count()
distinct_rows = payments_df.distinct().count()

print(f"Total Rows: {total_rows}")
print(f"Distinct Rows: {distinct_rows}")


I compared the total number of records with the number of distinct records and found no duplicate rows. This confirms that each payment transaction is uniquely stored and prevents issues such as duplicate payment amounts or incorrect financial reporting.

In [0]:
#Payment value should not be negative
payments_df.filter(col("payment_value") < 0).count()

In [0]:
payments_df.filter(col("payment_installments") < 0).count()

In [0]:
payments_df.filter(col("payment_type") == "").count()  #Payment type should not be empty

In [0]:
payments_df.filter(col("payment_value") <= 0).count() #Payment value should be greater than zero

There are 9 values with $0 as payments lets investigate them


In [0]:
payments_df.filter(col("payment_value") <= 0).show(truncate=False)

Most zero-value payment records are associated with the voucher payment type, suggesting that voucher-based transactions may explain these zero-value payments. However, some records have payment_type = not_defined, so the exact business reason cannot be confirmed from this dataset alone.

In [0]:
payments_df.filter(col("payment_installments") == 0).count() #installment should be atleast 1


In [0]:
#investigating these records
payments_df.filter(col("payment_installments") <= 0).show(truncate=False)

During business rule validation, I identified two credit card transactions with payment_installments = 0. Since credit card payments are generally expected to have at least one installment, I classified these as potential data quality issues. However, because no business rule specified how such records should be corrected, I retained them in the Silver layer and documented them for business review to preserve data integrity.

### IMP
I identified 9 records with a payment value of 0 and 2 credit card transactions with zero installments. Since no business rule specified that these records were invalid, I retained them in the Silver layer and documented them for further business review, ensuring data integrity while highlighting potential data quality issues.

In [0]:
#creating the bronze table
payments_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bronze_payments")

In [0]:
bronze_payments = spark.table("bronze_payments")
display(bronze_payments)

In [0]:
#creating the silver table
silver_payments_df = bronze_payments

In [0]:
silver_payments_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_payments")

In [0]:
silver_payments = spark.table("silver_payments")
display(silver_payments)

> We completed the **Bronze** and **Silver** layers of our ETL pipeline in Databricks using PySpark. The Bronze layer stores the raw data, while the Silver layer contains cleaned and validated data. Now, we're moving to the **Gold** layer to create business-ready tables and KPIs for reporting and analytics.


### Step 1 :-loading all the silver tables in their own Variable

In [0]:
# Load all Silver tables

silver_orders = spark.table("silver_orders")
silver_customers = spark.table("silver_customers")
silver_order_items = spark.table("silver_order_items")
silver_products = spark.table("silver_products")
silver_payments = spark.table("silver_payments")

In [0]:
display(silver_orders)
display(silver_customers)
display(silver_order_items)
display(silver_products)
display(silver_payments)

### Calculating total orders

In [0]:
from pyspark.sql.functions import count

total_orders = silver_orders.select(
    count("*").alias("total_orders")
)

display(total_orders)

### calculating total revenue

In [0]:
from pyspark.sql.functions import sum, round

total_revenue = silver_payments.select(
    round(sum("payment_value"), 2).alias("total_revenue")
)

display(total_revenue)

### On average, how much does a customer spend per order?


In [0]:
from pyspark.sql.functions import avg, round

average_order_value = silver_payments.select(
    round(avg("payment_value"), 2).alias("average_order_value")
)

display(average_order_value)

### finding the no of unique cust

In [0]:
 
from pyspark.sql.functions import countDistinct

total_customers = silver_orders.select(
    countDistinct("customer_id").alias("total_customers")   
)

display(total_customers)

### Total Products Sold

In [0]:
from pyspark.sql.functions import count

total_products_sold = silver_order_items.select(
    count("product_id").alias("total_products_sold")
)

display(total_products_sold)

### Combine all KPIs into one Gold DataFrame

In [0]:
#we will use join here now
from pyspark.sql.functions import lit

gold_sales_summary = (
    total_orders
    .crossJoin(total_revenue)
    .crossJoin(average_order_value)
    .crossJoin(total_customers)
    .crossJoin(total_products_sold)
)

display(gold_sales_summary)

### Saving the golds table

In [0]:
gold_sales_summary.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_sales_summary")

In [0]:
gold_sales_summary = spark.table("gold_sales_summary")

display(gold_sales_summary)

### gold_customer_summary

In [0]:
# loading the necassary tables
silver_orders = spark.table("silver_orders")
silver_payments = spark.table("silver_payments")
silver_order_items = spark.table("silver_order_items")

In [0]:
#joining the Three tables
customer_summary = (
    silver_orders
    .join(silver_payments, on="order_id", how="inner")
    .join(silver_order_items, on="order_id", how="inner")
)

In [0]:
display(customer_summary)

When we join two or more tables, some rows can be duplicated because of one-to-many relationships. We check the joined data to identify these duplicates before calculating KPIs so that our business metrics remain accurate.

In [0]:
from pyspark.sql.functions import count

customer_summary.groupBy("order_id").agg(
    count("*").alias("rows_after_join")
).show()

We can see that there are multiple records that got duplicated after the join which will give us wrong insights so instead od using this joined table we will perform our task without joining it 

### Customer-Level Insights
- we will look at cust specific insights

In [0]:
#joining the Two tables so that we can get the customer unique id which can be used to identify each cust has made how many orders
orders_customers = silver_orders.join(
    silver_customers.select("customer_id", "customer_unique_id"),
    on="customer_id",
    how="inner"
)

display(orders_customers)

In [0]:
from pyspark.sql.functions import count, col, round

# Counting how many customers fall into each order count
order_distribution = customer_orders.groupBy("total_orders").agg(
    count("*").alias("customer_count")
)

# Total number of unique customers
total_customers = customer_orders.count()

# Calculate percentage
order_distribution = order_distribution.withColumn(
    "percentage",
    round((col("customer_count") / total_customers) * 100, 2)
).orderBy("total_orders")

display(order_distribution)

# Business Insights

1. 96.88% of customers placed only one order.
2. Only 3.12% of customers made repeat purchases.
3. Customers with 5+ orders account for less than 0.02% of the customer base.

### 2.Calculate Total Amount Spent per Customer

In [0]:
#Since silver_payments col does not contain customer_unique_id, we first need to bring it in by joining with orders_customers
from pyspark.sql.functions import sum, round

customer_spending = (
    silver_payments
    .join(
        orders_customers.select("order_id", "customer_unique_id"),
        on="order_id",
        how="inner"
    )
    .groupBy("customer_unique_id")
    .agg(
        round(sum("payment_value"), 2).alias("total_amount_spent")
    )
)

display(customer_spending)

### 3. Top 10 Highest Spending Customers

In [0]:
from pyspark.sql.functions import sum, round, desc

top_10_customers = (
    silver_payments
    .join(
        orders_customers.select("order_id", "customer_unique_id"),
        on="order_id",
        how="inner"
    )
    .groupBy("customer_unique_id")
    .agg(
        round(sum("payment_value"), 2).alias("total_amount_spent")
    )
    .orderBy(desc("total_amount_spent"))
    .limit(10)
)

display(top_10_customers)

### IMP INSIGHT :-
By identifying the top 10 highest-spending customers, the company can reward them with exclusive offers, loyalty programs, or personalized benefits. This helps strengthen customer relationships, encourage repeat purchases, and enhance the company's brand loyalty and goodwill.

### 4. Top 10 Most Frequent Customers (By Number of Orders)

In [0]:
from pyspark.sql.functions import count, desc

top_10_frequent_customers = (
    orders_customers
    .groupBy("customer_unique_id")
    .agg(
        count("order_id").alias("total_orders")
    )
    .orderBy(desc("total_orders"))
    .limit(10)
)

display(top_10_frequent_customers)

### 5. Customer Segmentation Based on Total Spending

In [0]:
from pyspark.sql.functions import when

customer_segments = customer_spending.withColumn(
    "customer_segment",
    when(customer_spending.total_amount_spent >= 1000, "VIP")
    .when(customer_spending.total_amount_spent >= 500, "High Value")
    .when(customer_spending.total_amount_spent >= 200, "Medium Value")
    .otherwise("Low Value")
)

display(customer_segments)

In [0]:
from pyspark.sql.functions import count, col, round

segment_distribution = customer_segments.groupBy("customer_segment").agg(
    count("*").alias("customer_count")
)

total_customers = customer_segments.count()

segment_distribution = segment_distribution.withColumn(
    "percentage",
    round((col("customer_count") / total_customers) * 100, 2)
).orderBy(col("percentage").desc())

display(segment_distribution)

### IMP INSIGHT:-
Customer segmentation revealed that 78.41% of customers belong to the Low Value segment, while only 1.27% are VIP customers. These insights can help the business design targeted marketing campaigns, improve customer retention, and increase customer lifetime value by focusing on moving customers into higher-value segments.

### 6. Tp 10 Product Categories by Revenue

###

In [0]:
from pyspark.sql.functions import sum, round, desc

top_product_categories = (
    silver_order_items
    .join(
        silver_products.select("product_id", "product_category_name"),
        on="product_id",
        how="inner"
    )
    .groupBy("product_category_name")
    .agg(
        round(sum("price"), 2).alias("total_revenue")
    )
    .orderBy(desc("total_revenue"))
    .limit(10)
)

display(top_product_categories)

Identified the top 10 product categories by revenue, enabling the business to focus marketing efforts, inventory planning, and strategic investments on the categories that contribute the most to overall sales.

### 7.Top 10 Least Revenue-Generating Product Categories

In [0]:
from pyspark.sql.functions import sum, round, asc

least_product_categories = (
    silver_order_items
    .join(
        silver_products.select("product_id", "product_category_name"),
        on="product_id",
        how="inner"
    )
    .groupBy("product_category_name")
    .agg(
        round(sum("price"), 2).alias("total_revenue")
    )
    .orderBy(asc("total_revenue"))
    .limit(10)
)

display(least_product_categories)

Identified the 10 lowest revenue-generating product categories to highlight underperforming areas of the business. These insights can support decisions related to product optimization, inventory management, and targeted promotional campaigns.

### Final Project Summary

Successfully built an end-to-end ETL pipeline using the **Bronze, Silver, and Gold** architecture in Databricks with PySpark. The project transformed raw data into business-ready datasets and generated two Gold tables (`gold_sales_summary` and `gold_customer_summary`) to support reporting. Key insights included customer purchase behavior, spending patterns, customer segmentation, and the highest and lowest revenue-generating product categories, demonstrating how data engineering can drive meaningful business decisions.
